# The Whole Cycle, On Your Own

**Notebook 5 of 5.** Nothing new is taught. You run the whole workflow yourself on the
concrete data — split, baseline, tune, score once, then look at what went wrong and what the
model leaned on.

One difference: notebooks 02 and 03 chose `max_depth` by watching the test score, which 02
said plainly is not allowed. Here the choice is made with **cross-validation on the training
set**, and the test set is touched once, at the end.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

## 1. Load and split

> **Exercise:**
>
> Read `data/concrete.csv`, separate `compressive_strength` as the target, and split off 20%
> as a test set with `random_state=42`.

<details><summary>
Click here for a hint...
</summary>

`train_test_split(X, y, test_size=0.2, random_state=42)`. No `stratify`: the target is
continuous.

</details>

In [ ]:
df = pd.read_csv("../data/concrete.csv")

X = df.drop(columns="compressive_strength")
y = df["compressive_strength"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"{len(X_train)} training mixes, {len(X_test)} test mixes")

## 2. The number to beat

> **Exercise:**
>
> Fit a `DummyRegressor` and report its test MAE and R². Anything you build has to beat this.

<details><summary>
Click here for a hint...
</summary>

`DummyRegressor()` predicts the training mean by default.

</details>

In [ ]:
# predicts the training mean for every mix, whatever its recipe
baseline = DummyRegressor().fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print(f"baseline MAE: {mean_absolute_error(y_test, baseline_pred):.2f} MPa")
print(f"baseline R²:  {r2_score(y_test, baseline_pred):.3f}")

<details><summary>
Click here to compare your answer...
</summary>

**MAE 13.05 MPa, R² -0.000.** Predicting the mean misses by 13 MPa on average and explains
none of the variance, which is what R² = 0 means by construction.

</details>

## 3. Choosing a depth without touching the test set

Cross-validation splits the **training** set into folds, fits on all but one, scores the one
left out, and averages — an honest estimate while the test set stays sealed.

> **Exercise:**
>
> For `max_depth` 1 to 15, run 5-fold cross-validation on the training set with
> `scoring="neg_root_mean_squared_error"` and plot the mean RMSE. Which depth wins?

<details><summary>
Click here for a hint...
</summary>

`cv = KFold(n_splits=5, shuffle=True, random_state=42)`, then `cross_val_score(...)`. Scores
come back **negative** — flip the sign.

</details>

In [ ]:
# the same five folds for every depth, so the comparison is fair
cv = KFold(n_splits=5, shuffle=True, random_state=42)
depths = range(1, 16)
cv_rmse = []

for depth in depths:

    # five fits per depth, each scored on the fold it did not see
    scores = cross_val_score(
        DecisionTreeRegressor(max_depth=depth, random_state=42),
        X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error",
    )

    # scikit-learn returns the negative RMSE, so flip the sign back
    cv_rmse.append(-scores.mean())

plt.figure(figsize=(8, 5))
plt.plot(depths, cv_rmse, marker="o", color="#FF4A11")
plt.xlabel("max_depth"); plt.ylabel("cross-validated RMSE (MPa)")
plt.xticks(list(depths)); plt.title("Five-fold cross-validation on the training set")
plt.show()

print(f"best depth: {depths[int(np.argmin(cv_rmse))]}, CV RMSE {min(cv_rmse):.2f}")

<details><summary>
Click here to compare your answer...
</summary>

**Depth 10, CV RMSE 6.96**, though everything from depth 8 up sits between 6.9 and 7.2. The
curve drops steeply to about depth 7, then flattens, and the gaps between neighboring depths
are smaller than the spread across folds.

That flatness is information: the choice is not critical here, so do not agonize over it.

</details>

## 4. Two hyperparameters at once

`GridSearchCV` runs the same cross-validation over every combination and keeps the best.

> **Exercise:**
>
> Search `max_depth` over `[4, 6, 8, 10, 12, None]` and `min_samples_leaf` over
> `[1, 2, 5, 10, 20]`, same folds. Print the winning combination and its CV RMSE.

<details><summary>
Click here for a hint...
</summary>

`GridSearchCV(..., cv=cv, scoring="neg_root_mean_squared_error")`, then `.best_params_` and
`-.best_score_`.

</details>

In [ ]:
# every combination of the two grids, cross-validated on the training set alone
grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    {"max_depth": [4, 6, 8, 10, 12, None], "min_samples_leaf": [1, 2, 5, 10, 20]},
    cv=cv,
    scoring="neg_root_mean_squared_error",
)
grid.fit(X_train, y_train)

print(f"best parameters: {grid.best_params_}")
print(f"CV RMSE: {-grid.best_score_:.2f}")

<details><summary>
Click here to compare your answer...
</summary>

**`max_depth=None`, `min_samples_leaf=2`, CV RMSE 6.73** — better than any depth limit alone.

No depth limit at all, with growth stopped a different way. The two are not interchangeable,
and searching them together found what neither search alone would.

</details>

## 5. The test set, once

> **Exercise:**
>
> Score `grid.best_estimator_` on the test set: MAE, RMSE and R². Compare with the baseline.

<details><summary>
Click here for a hint...
</summary>

`GridSearchCV` already refitted the best model on the whole training set, so
`.predict(X_test)` is all that is needed.

</details>

In [ ]:
# already refitted on the whole training set, so it is ready to score
model = grid.best_estimator_
predictions = model.predict(X_test)

print(f"MAE:  {mean_absolute_error(y_test, predictions):.2f} MPa   (baseline 13.05)")
print(f"RMSE: {root_mean_squared_error(y_test, predictions):.2f} MPa")
print(f"R²:   {r2_score(y_test, predictions):.3f}   (baseline 0.000)")

<details><summary>
Click here to compare your answer...
</summary>

**MAE 4.66 MPa, RMSE 6.95, R² 0.812.** A third of the baseline's miss, and 81% of the
variance explained.

Test RMSE (6.95) landed close to the cross-validated estimate (6.73), which is what you want:
the tuning did not quietly fit the folds.

</details>

## 6. Where it goes wrong

One score hides which mixes the model handles badly.

> **Exercise:**
>
> Build a frame of true strength, prediction and absolute error for every test mix. Print the
> ten worst, then the mean absolute error within each quartile of true strength. What pattern
> do you see?

<details><summary>
Click here for a hint...
</summary>

`pd.qcut(errors["true"], 4)` makes four equal-sized groups; group by it and average
`abs_error`.

</details>

In [ ]:
errors = pd.DataFrame({
    "true": y_test,
    "predicted": predictions,
    "abs_error": (y_test - predictions).abs(),
})

print(errors.nlargest(10, "abs_error").round(2).to_string())

# four groups of equal size, split by true strength rather than by error
quartiles = pd.qcut(errors["true"], 4)
print()
print(errors.groupby(quartiles, observed=True)["abs_error"].mean().round(2))

<details><summary>
Click here to compare your answer...
</summary>

**The strong mixes are the hard ones**: 2.43 MPa average miss in the weakest quartile, 7.70
in the strongest. The ten worst average 47.7 MPa of true strength against 35.7 overall.

The reason is in the algorithm. A leaf predicts the **mean** of its training mixes, and means
are pulled toward the middle, so extremes are systematically undershot — the same flattening
you saw at the edge of the data in notebook 03, here in the middle of it.

Predictions also repeat in the worst-ten list: the dataset has 25 exact duplicates and many
near-duplicate recipes, so a mistake is often made twice.

</details>

## 7. What the model leaned on

> **Exercise:**
>
> Run permutation importance on the tuned model with the test set, `n_repeats=10`,
> `random_state=42`, `scoring="r2"`, and rank the features.

<details><summary>
Click here for a hint...
</summary>

`.importances_mean` into a `Series` indexed by `X.columns`.

</details>

In [ ]:
# one column scrambled at a time, ten times each, on the test set
importance = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42, scoring="r2"
)

pd.Series(importance.importances_mean, index=X.columns).sort_values(ascending=False).round(3)

<details><summary>
Click here to compare your answer...
</summary>

**Age 0.816, cement 0.736, water 0.422, slag 0.387**, the aggregates far behind. Scrambling
age costs four fifths of the R².

Concrete cures over time and cement is what hardens: the model found the two things a
material scientist would have named first.

</details>

## Summary

The cycle, end to end:

1. **Split** before anything else.
2. **Baseline** first, so improvement means something: MAE 13.05.
3. **Tune with cross-validation on the training set**: no depth limit, leaves of at least 2,
   CV RMSE 6.73.
4. **Touch the test set once**: MAE 4.66, R² 0.812.
5. **Look at the errors**, not only the score. Strong mixes are undershot, because leaves
   predict means.
6. **Check what it leaned on**: age and cement.

This is the shape of every modeling task. What changes is the model in step 3 — and a single
tree, for all its readability, is about as good as a single tree gets.

A single tree is where this stops, and it is not where the week stops: one tree is sensitive
to small changes in its data, as the boundary in notebook 01 showed. What follows is built
out of many of them.